# Vector Database

In [ ]:
# Data
import os
import pandas as pd
import kagglehub

path = kagglehub.dataset_download("gpreda/bbc-news")
news_data = pd.read_csv(os.path.join(path, "bbc_news.csv"))
news_data.head()

In [ ]:
# Connect to Weaviate Cloud
import weaviate
from weaviate.classes.config import Configure, Property, DataType
from weaviate.classes.query import Filter, Rerank

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=os.environ["WEAVIATE_API_KEY"]
)

In [ ]:
# Import and Vectorize objects
if client.collections.exists("bbc_collection"):
    client.collections.delete("bbc_collection")

news_collection = client.collections.create(
    name="bbc_collection",
    vector_config=Configure.Vectors.text2vec_weaviate(),
    properties=[
        Property(
            name="description",
            data_type=DataType.TEXT,
            module_config=Configure.NamedVectors.text2vec_weaviate("description")
        )
    ]
)

data_objects = news_data.to_dict(orient="records")

news_collection = client.collections.use("bbc_collection")
with news_collection.batch.fixed_size(batch_size=200) as batch:
    for obj in data_objects:
        batch.add_object(properties=obj)

### Using Weaviate Cloud

In [ ]:
# Fetch
object = news_collection.query.fetch_objects(limit = 1, include_vector = True).objects[0]
object.properties

In [ ]:
# Vector size
len(object.vector["default"])

In [ ]:
# Keyword Search
response = news_collection.query.bm25("Hong Kong", limit=5)
pd.DataFrame([o.properties for o in response.objects])

In [ ]:
# Semantic Search
response = news_collection.query.near_text("Hong Kong", limit=5)
pd.DataFrame([o.properties for o in response.objects])

In [ ]:
# Hybrid Search
response = news_collection.query.hybrid("Hong Kong", alpha=0.2, limit = 5)
pd.DataFrame([o.properties for o in response.objects])

In [ ]:
# Metadata Filtering
news_collection = client.collections.get("bbc_collection")
where_filter = Filter.by_property("description").contains_any(["Canada", "Hong Kong"])
response = news_collection.query.fetch_objects(filters=where_filter, limit=5, include_vector=True)
pd.DataFrame([o.properties for o in response.objects])

In [ ]:
# TODO: Reranking

In [ ]:
client.close()